# Campanha 002 — Qwen3-4B LoRA + GGUF (Kaggle GPU)Notebook **autocontido** para treinar o adaptador LoRA do `Qwen/Qwen3-4B` na **Campanha 002**e exportar `GGUF (q4_k_m)`. Espelha a receita do worker COSCA:- `laboratory/train/worker_self_contained.py` (treino — lógica canônica)- `laboratory/train/merge_gguf.py` (merge em FP16 + GGUF)> **Fluxo:** setup deps -> dataset embutido (inline) -> LoRA/QLoRA (4-bit) -> train -> merge (FP16) -> GGUF.> Output final: `/kaggle/working/output/campaign-002-qwen3-4b-lora.gguf`**Acelerador recomendado:** GPU T4x2 ou P100 (16 GB). `CUDA_VISIBLE_DEVICES=0` (1 GPU — suficiente p/ QLoRA 4B com gradient checkpointing).---> ⚠️ **Aviso honesto (OOM):** `max_seq_len=8192` é o TETO de truncamento do `SFTConfig` (`max_length`),> **não** o comprimento real dos exemplos. Os 15 exemplos SFT são curtos (poucas centenas de tokens),> então o treino usa o comprimento real de cada exemplo e não há pad até 8192 — risco de OOM é baixo> com `use_gradient_checkpointing="unsloth"`. **Se** um exemplo longo for adicionado e estourar memória,> reduza a variável `MAX_SEQ` (ex.: `2048`) na célula de treino. Veja `kaggle_HOWTO.md`.

In [ ]:
# [kaggle] SETUP — instala a stack de treino (espelha worker_self_contained.install_deps)
import os, sys, subprocess, platform
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # usa 1 GPU (T4/P100) — suficiente p/ QLoRA de 4B
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

# Só instala o que não está presente / força a versão compatível com unsloth.
pkgs = ["unsloth", "trl", "transformers", "datasets", "peft", "accelerate", "bitsandbytes", "einops"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *pkgs])
if r.returncode != 0:
    print("[kaggle] pip retornou exit=%s — tentando instalar um a um..." % r.returncode)
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", p])

# Diagnóstico de ambiente
try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            print("  GPU[%d] = %s" % (i, torch.cuda.get_device_name(i)))
        print("  Vram total GPU0 (GB): %.1f" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
except Exception as e:
    print("[kaggle] torch não importou ainda:", e)
    print("[kaggle] Rode `!pip install torch` e reinicie o kernel se persistir.")
print("Done: setup")


In [ ]:
# [kaggle] DATA — dataset embutido INLINE (base64, igual ao worker.materialize)
# O notebook é autocontido: não depende de upload de dataset externo.
import base64, json, os

TRAIN_B64 = "..."
VAL_B64   = "eyJtZXNzYWdlcyI6W3sicm9sZSI6InN5c3RlbSIsImNvbnRlbnQiOiJZb3UgYXJlIGEgY29kaW5nIGFnZW50IGluIGEgd29ya3NwYWNlLiBZb3UgTVVTVCB1c2UgdG9vbHMgdG8gYWNjb21wbGlzaCB0aGUgdGFzay4gQWx3YXlzIHJlYWRfZmlsZSBiZWZvcmUgZWRpdF9maWxlLiBvbGRfc3RyaW5nIG11c3QgYmUgYW4gZXhhY3Qgc3Vic3RyaW5nIHlvdSBvYnNlcnZlZC4gSWYgYSB0b29sIGVycm9ycywgcmVhZCBhZ2FpbiBhbmQgcmV0cnkuIEV4ZWN1dGUgYnkgY2FsbGluZyB0b29scywgbm90IGRlc2NyaWJpbmcuIn0seyJyb2xlIjoidXNlciIsImNvbnRlbnQiOiJVcGRhdGUgbWFpbi5nbzogSSdtIG5vdCBzdXJlIHdoZXJlIHRvIHBsYWNlIHRoZSBuZXcgaGFuZGxlci4gRmluZCB0aGUgcmlnaHQgZmlsZSBmaXJzdC5cblxuSW5pdGlhbCB3b3Jrc3BhY2U6XG4tLS0gbWFpbi5nbyAtLS1cbnBhY2thZ2UgbWFpblxuXG5mdW5jIFJlc3RhcnQoKSB7XG5cdC8vIG9sZFxufVxuIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgbGlzdF9kaXIoe1wicGF0aFwiOiBcIlwifSkifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiQ2FsbCByZWFkX2ZpbGUoe1wicGF0aFwiOiBcIm1haW4uZ29cIn0pIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgZWRpdF9maWxlKHtcIm5ld19zdHJpbmdcIjogXCJmdW5jIFJlc3RhcnQoKSB7XFxuXFxuLy8gTmV3IGhhbmRsZXIgYWRkZWRcXG5mdW5jIE5ld0hhbmRsZXIoKSB7XFxuICAgIC8vIGhhbmRsZXIgbG9naWMgaGVyZVxcbn1cIiwgXCJwYXRoXCI6IFwibWFpbi5nb1wiLCBcIm9sZF9zdHJpbmdcIjogXCJmdW5jIFJlc3RhcnQoKSB7XCJ9KSJ9LHsicm9sZSI6ImFzc2lzdGFudCIsImNvbnRlbnQiOiJUaGUgbWFpbi5nbyBmaWxlIGhhcyBiZWVuIHVwZGF0ZWQgc3VjY2Vzc2Z1bGx5IHdpdGggdGhlIG5ldyBoYW5kbGVyIGZ1bmN0aW9uLiBUaGUgZmlsZSBub3cgY29udGFpbnMgYm90aCB0aGUgZXhpc3RpbmcgUmVzdGFydCBmdW5jdGlvbiBhbmQgdGhlIG5ld2x5IGFkZGVkIE5ld0hhbmRsZXIgZnVuY3Rpb24uIn1dfQp7Im1lc3NhZ2VzIjpbeyJyb2xlIjoic3lzdGVtIiwiY29udGVudCI6IllvdSBhcmUgYSBjb2RpbmcgYWdlbnQgaW4gYSB3b3Jrc3BhY2UuIFlvdSBNVVNUIHVzZSB0b29scyB0byBhY2NvbXBsaXNoIHRoZSB0YXNrLiBBbHdheXMgcmVhZF9maWxlIGJlZm9yZSBlZGl0X2ZpbGUuIG9sZF9zdHJpbmcgbXVzdCBiZSBhbiBleGFjdCBzdWJzdHJpbmcgeW91IG9ic2VydmVkLiBJZiBhIHRvb2wgZXJyb3JzLCByZWFkIGFnYWluIGFuZCByZXRyeS4gRXhlY3V0ZSBieSBjYWxsaW5nIHRvb2xzLCBub3QgZGVzY3JpYmluZy4ifSx7InJvbGUiOiJ1c2VyIiwiY29udGVudCI6IlVwZGF0ZSBtYWluLmpzb246IEknbSBub3Qgc3VyZSB3aGVyZSB0byBwbGFjZSB0aGUgbmV3IGhhbmRsZXIuIEZpbmQgdGhlIHJpZ2h0IGZpbGUgZmlyc3QuXG5cbkluaXRpYWwgd29ya3NwYWNlOlxuLS0tIG1haW4uanNvbiAtLS1cbntcbiAgXCJyZXN0YXJ0XCI6IHtcbiAgICBcIm1vZGVcIjogXCJjb2xkXCJcbiAgfVxufVxuIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgbGlzdF9kaXIoe1wicGF0aFwiOiBcInJvb3RcIn0pIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgZ2xvYih7XCJwYXRoXCI6IFwicm9vdFwiLCBcInBhdHRlcm5cIjogXCJtYWluLmpzb25cIn0pIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgbGlzdF9kaXIoe1wicGF0aFwiOiBcIi5cIn0pIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgcmVhZF9maWxlKHtcInBhdGhcIjogXCJtYWluLmpzb25cIn0pIn0seyJyb2xlIjoiYXNzaXN0YW50IiwiY29udGVudCI6IkNhbGwgZWRpdF9maWxlKHtcInBhdGhcIjogXCJtYWluLmpzb25cIiwgXCJvbGRfc3RyaW5nXCI6IFwicmVzdGFydFwiLCBcIm5ld19zdHJpbmdcIjogXCJyZXN0YXJ0XFxuXFxcImhhbmRsZXJzXFxcIjoge1xcXCJleGFtcGxlXFxcIjogXFxcImhhbmRsZXJcXFwifVwifSkifSx7InJvbGUiOiJhc3Npc3RhbnQiLCJjb250ZW50IjoiVGhlIHJpZ2h0IGZpbGUgdG8gdXBkYXRlIGlzIGBtYWluLmpzb25gLiBJIHJlYWQgaXRzIGN1cnJlbnQgY29udGVudCwgd2hpY2ggaXMgYHtcInJlc3RhcnRcIjoge1wibW9kZVwiOiBcImNvbGRcIn19YCwgYW5kIHN1Y2Nlc3NmdWxseSBhZGRlZCB0aGUgbmV3IGhhbmRsZXIgc3RydWN0dXJlIGJ5IGVkaXRpbmcgdGhlIGZpbGUuIFRoZSB1cGRhdGVkIGZpbGUgbm93IGNvbnRhaW5zOlxuXG5gYGBqc29uXG57XG4gIFwicmVzdGFydFwiOiB7XG4gICAgXCJtb2RlXCI6IFwiY29sZFwiXG4gIH0sXG4gIFwiaGFuZGxlcnNcIjoge1xuICAgIFwiZXhhbXBsZVwiOiBcImhhbmRsZXJcIlxuICB9XG59XG5gYGAifV19Cg=="

os.makedirs("/kaggle/working", exist_ok=True)
train_path = "/kaggle/working/dataset-curado-002.train.jsonl"
val_path   = "/kaggle/working/dataset-curado-002.val.jsonl"
with open(train_path, "wb") as f:
    f.write(base64.b64decode(TRAIN_B64))
with open(val_path, "wb") as f:
    f.write(base64.b64decode(VAL_B64))

def load_records(path):
    recs = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    return recs

train_records = load_records(train_path)
val_records   = load_records(val_path)
print("train exemplos:", len(train_records))
print("val   exemplos:", len(val_records))

# Exibe o primeiro exemplo (controle), como faz o worker no format_dataset.
if train_records:
    msgs = train_records[0]["messages"]
    print("roles no 1o exemplo:", [m["role"] for m in msgs])
    print("1o user content (primeiros 120 chars):", msgs[0]["content"][:120])


In [ ]:
# [kaggle] TRAIN — carrega base em 4-bit, aplica LoRA, treina (espelha worker_self_contained)
import os, json
os.makedirs("/kaggle/working", exist_ok=True)

BASE      = "Qwen/Qwen3-4B"
ADAPTER   = "/kaggle/working/adapter-campaign-002"
MAX_SEQ   = 8192        # [kaggle] teto de truncamento. Se OOM no T4 (16GB), reduza p/ 2048.
EPOCHS    = 3
LR        = 2e-4
BATCH     = 2
GRAD_ACC  = 4
LORA_R    = 16
LORA_ALPHA= 32
LORA_TGT  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

# Dataset -> records (igual worker)
train_path = "/kaggle/working/dataset-curado-002.train.jsonl"
records = []
with open(train_path, "r", encoding="utf-8") as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))
if not records:
    raise RuntimeError("Dataset vazio.")
train_dataset = Dataset.from_list(records)
print("dataset:", len(train_dataset), "exemplos")

# ------------------------- helpers (do worker) -------------------------
def manual_chat_template(messages):
    out = []
    for m in messages:
        out.append("<|im_start|>%s\n%s<|im_end|>\n" % (m["role"], m["content"]))
    return "".join(out)

def install_eos_compat(tokenizer):
    """Mapeia <EOS_TOKEN> -> eos real (sem redimensionar embeddings)."""
    real_eos = tokenizer.eos_token
    real_eos_id = tokenizer.eos_token_id
    original = tokenizer.convert_tokens_to_ids
    def _c(tokens):
        if isinstance(tokens, str):
            return real_eos_id if tokens == "<EOS_TOKEN>" else original(tokens)
        if isinstance(tokens, (list, tuple)):
            return [real_eos_id if t == "<EOS_TOKEN>" else original(t) for t in tokens]
        return original(tokens)
    tokenizer.convert_tokens_to_ids = _c
    print("EOS compat: <EOS_TOKEN> -> %r (id=%s)" % (real_eos, real_eos_id))
    return tokenizer

def format_dataset(dataset, tokenizer):
    def _fmt(ex):
        messages = ex["messages"]
        if getattr(tokenizer, "chat_template", None):
            try:
                text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
                return {"text": text}
            except Exception as e:
                print("AVISO: chat_template falhou; fallback manual:", e)
        return {"text": manual_chat_template(messages)}
    cols = [c for c in dataset.column_names if c != "text"]
    return dataset.map(_fmt, remove_columns=cols, desc="Formatando dataset COSCA")

# ------------------------- MODEL (4-bit QLoRA) -------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)

# tokenizer/model eos-pad (validate_tokenizer do worker)
eos = tokenizer.eos_token
eos_id = tokenizer.eos_token_id
if eos_id is None:
    raise RuntimeError("eos_token_id is None.")
try:
    assert eos in tokenizer.get_vocab()
except Exception as e:
    raise RuntimeError("EOS nao esta no vocab: %s" % e)
tokenizer.pad_token = eos
model.config.eos_token_id = eos_id
model.config.pad_token_id = eos_id
if hasattr(model, "generation_config"):
    model.generation_config.eos_token_id = eos_id
    model.generation_config.pad_token_id = eos_id
print("EOS=%r id=%s pad=%r" % (eos, eos_id, tokenizer.pad_token))

tokenizer = install_eos_compat(tokenizer)
if tokenizer.convert_tokens_to_ids("<EOS_TOKEN>") != tokenizer.eos_token_id:
    raise RuntimeError("EOS compatibility falhou.")

# LoRA
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, target_modules=LORA_TGT, lora_alpha=LORA_ALPHA,
    lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=42)

train_dataset = format_dataset(train_dataset, tokenizer)
if len(train_dataset) > 0:
    first = train_dataset[0]["text"]
    print("primeiro exemplo (prévia 600):\n", first[:600])
    print("contem <EOS_TOKEN>:", "<EOS_TOKEN>" in first)
    print("contem <|im_end|>:", "<|im_end|>" in first)

# ------------------------- SFT CONFIG (shim p/ TRL) -------------------------
# [kaggle] O worker usa max_length + eos_token. TRL mudou o nome de max_seq_length:
# este shim tenta max_length e cai para a assinatura que a versão instalada aceitar.
def build_sft_config(real_eos):
    common = dict(
        output_dir=ADAPTER,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACC,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="linear",
        warmup_steps=10,
        max_length=MAX_SEQ,          # worker usa max_length
        eos_token=real_eos,          # worker usa eos_token
        logging_steps=5,
        save_strategy="epoch",
        report_to="none",
        bf16=False,
        fp16=True,
        seed=42,
        data_seed=42,
    )
    try:
        return SFTConfig(**common)
    except TypeError:
        # fallback compat: SFTConfig pode não aceitar max_length/eos_token
        if "max_seq_length" not in common and "max_length" in common:
            common["max_seq_length"] = common.pop("max_length")
        common.pop("eos_token", None)
        return SFTConfig(**common)

sft_cfg = build_sft_config(eos)
print("SFTConfig ok. max_length=%s eos=%r" % (getattr(sft_cfg, "max_length", getattr(sft_cfg, "max_seq_length", "?")), getattr(sft_cfg, "eos_token", None)))

# SFTTrainer — [kaggle] shim p/ apis antigas (tokenizer=) e novas (processing_class=)
os.makedirs(ADAPTER, exist_ok=True)
sft_kwargs = dict(model=model, train_dataset=train_dataset, args=sft_cfg)
try:
    trainer = SFTTrainer(**sft_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = SFTTrainer(**sft_kwargs, tokenizer=tokenizer)
print("SFTTrainer criado.")

print("[kaggle] TREINANDO...")
train_result = trainer.train()
print("[kaggle] trainer.train() terminou.")

# ------------------------- ARTIFACT (adapter LoRA) -------------------------
os.makedirs(ADAPTER, exist_ok=True)
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print("[kaggle] adapter salvo em:", ADAPTER)
print("arquivos:", sorted(os.listdir(ADAPTER)))


In [ ]:
# [kaggle] MERGE + GGUF — base em FP16 (NÃO 4-bit), aplica adapter, funde, exporta q4_k_m.
# Espelha laboratório/train/merge_gguf.py.
import os, glob, shutil, subprocess, sys
import torch

BASE    = "Qwen/Qwen3-4B"
ADAPTER = "/kaggle/working/adapter-campaign-002"
MERGED  = "/kaggle/working/merged-campaign-002"
GGUFDIR = "/kaggle/working/gguf-campaign-002"
OUTPUT  = "/kaggle/working/output"
os.makedirs(OUTPUT, exist_ok=True)

if not os.path.exists(os.path.join(ADAPTER, "adapter_model.safetensors")):
    raise RuntimeError("adapter_model.safetensors nao encontrado em %s — rode a celula de treino antes." % ADAPTER)

from transformers import AutoTokenizer
from peft import PeftModel
from unsloth import FastLanguageModel

print("[kaggle] base (FP16):", BASE)
# PRINCÍPIO (merge_gguf.py): fundir em FP16, NUNCA em 4-bit.
# [kaggle] T4/P100 não suportam bf16 -> dtype explícito fp16.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=8192, dtype=torch.float16, load_in_4bit=False)

# eos/pad consistentes com o treino (usa o eos real do tokenizer de base)
real_eos = tokenizer.eos_token or "<|im_end|>"
tokenizer.eos_token = real_eos
tokenizer.pad_token = real_eos
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.eos_token_id
print("EOS:", real_eos, "id:", tokenizer.eos_token_id)

print("[kaggle] aplicando adapter LoRA...")
model = PeftModel.from_pretrained(model, ADAPTER)

print("[kaggle] fazendo MERGE (base + LoRA)...")
model = model.merge_and_unload()

# salva HF fundido (para fallback de conversão e para referência)
os.makedirs(MERGED, exist_ok=True)
model.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print("[kaggle] modelo fundido salvo em:", MERGED)

# ------------------------- Export GGUF (q4_k_m) -------------------------
os.makedirs(GGUFDIR, exist_ok=True)
gguf_files = []
try:
    # unsloth save_pretrained_gguf — [kaggle] shim de assinatura (tokenizer opcional)
    try:
        model.save_pretrained_gguf(GGUFDIR, tokenizer, quantization_method="q4_k_m")
    except TypeError:
        model.save_pretrained_gguf(GGUFDIR, quantization_method="q4_k_m")
    gguf_files = glob.glob(os.path.join(GGUFDIR, "*.gguf"))
    print("[kaggle] GGUF via unsloth:", gguf_files)
except Exception as e:
    print("[kaggle] unsloth save_pretrained_gguf falhou (%s) -> fallback llama.cpp convert" % e)
    gguf_files = []

# Fallback: llama.cpp convert_hf_to_gguf.py (python puro, roda em CPU)
if not gguf_files:
    LLAMA = "/kaggle/working/llama.cpp"
    conv = os.path.join(LLAMA, "convert_hf_to_gguf.py")
    if not os.path.exists(conv):
        print("[kaggle] clonando llama.cpp (depth 1)...")
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                        "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf", "sentencepiece", "protobuf"])
    outfile = os.path.join(OUTPUT, "campaign-002-qwen3-4b-lora.gguf")
    print("[kaggle] convertendo HF -> GGUF (q4_k_m)...")
    subprocess.run([sys.executable, conv, MERGED, "--outfile", outfile, "--outtype", "q4_k_m"], check=True)
    gguf_files = [outfile]
    print("[kaggle] GGUF via llama.cpp:", gguf_files)

print("[kaggle] GGUF final:", gguf_files)
for f in gguf_files:
    print("  %s (%.1f MB)" % (f, os.path.getsize(f) / 1e6))


In [ ]:
# [kaggle] SAVE — copia GGUF + adapter + report para /kaggle/working/output/
import os, glob, json, shutil, datetime
OUTPUT = "/kaggle/working/output"
GGUFDIR = "/kaggle/working/gguf-campaign-002"
ADAPTER = "/kaggle/working/adapter-campaign-002"
os.makedirs(OUTPUT, exist_ok=True)

name = "campaign-002-qwen3-4b-lora.gguf"
dest = os.path.join(OUTPUT, name)
candidates = glob.glob(os.path.join(GGUFDIR, "*.gguf")) + glob.glob(os.path.join(OUTPUT, "*.gguf"))
found = False
for p in candidates:
    if p.lower().endswith(".gguf"):
        shutil.copy2(p, dest)
        found = True
        print("[kaggle] GGUF copiado ->", dest, "(%.1f MB)" % (os.path.getsize(dest) / 1e6))
        break
if not found:
    raise RuntimeError("Nenhum .gguf encontrado — verifique a célula de merge.")

# Copia o adaptador LoRA como referência (não altera o treinado)
adapter_dst = os.path.join(OUTPUT, "adapter-campaign-002")
if os.path.exists(adapter_dst):
    shutil.rmtree(adapter_dst, ignore_errors=True)
shutil.copytree(ADAPTER, adapter_dst)

# WorkerReport (espelha WORKER_REPORT.json do worker)
report = {
    "campaign_id": "campaign-002",
    "status": "TRAINING_COMPLETE",
    "base_model": "Qwen/Qwen3-4B",
    "adapter": ADAPTER,
    "merged": "/kaggle/working/merged-campaign-002",
    "gguf": name,
    "quant": "q4_k_m",
    "metrics": {"epochs": 3, "lora_r": 16, "lora_alpha": 32},
    "policy": {"promotion": "COSCA_ONLY", "golden": "COSCA_ONLY"},
    "finished_at": datetime.datetime.now().isoformat(),
}
with open(os.path.join(OUTPUT, "WORKER_REPORT.json"), "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=2, ensure_ascii=False)
print("[kaggle] WORKER_REPORT.json escrito.")

print("\n=== ARQUIVOS EM OUTPUT (%s) ===" % OUTPUT)
for fn in sorted(os.listdir(OUTPUT)):
    fp = os.path.join(OUTPUT, fn)
    if os.path.isfile(fp):
        print("  %-40s %.2f MB" % (fn, os.path.getsize(fp) / 1e6))
    else:
        print("  %-40s (dir)" % fn)

print("\n[kaggle] BAIXE do Kaggle: /kaggle/working/output/campaign-002-qwen3-4b-lora.gguf")


## ResultadoTudo salvo em **`/kaggle/working/output/`**:| Arquivo | Descrição || --- | --- || `campaign-002-qwen3-4b-lora.gguf` | Modelo fundido (base Qwen3-4B + LoRA) quantizado em `q4_k_m` — **é este que o Don baixa** || `adapter-campaign-002/` | Adaptador LoRA (safetensors + config) — referência / reuso || `WORKER_REPORT.json` | Relatório de conclusão do worker |### Como baixar (Kaggle browser)No painel do notebook aberto, painel à direita **Output** → clique em `campaign-002-qwen3-4b-lora.gguf` → **Download** (ou marque + `Download selected`).### Próximos passos (no PC do Don)1. Baixe o `.gguf` para `laboratory/campaign-002/output/`.2. Crie o `Modelfile` do Ollama baseado no `Qwen3-4B` (base = modelo fundido).3. Carregue como `cosca-qwen3-4b-lora-002`.4. Rode a validação AFTER do COSCA (golden gate) — o worker NÃO promove; o COSCA valida.> ⚙️ Cada célula com `# [kaggle]` é o único ponto que pode precisar de ajuste manual.> Para **automação** via API (`kaggle kernels push`) seria preciso `KAGGLE_USERNAME`/`KAGGLE_KEY`;> o caminho default é rodar no browser. Veja `kaggle_HOWTO.md`.